# H1 ABSA-RTS Pipeline (Colab)

This notebook runs the **H1 negation/contrast diagnostics** pipeline end-to-end:
1. Build ABSA-RTS pairs/eval set
2. Generate A1 prediction file (placeholder predictor with configurable noise)
3. Evaluate A0 vs A1 and export metrics


## 0) Set project path

- If you cloned the repository in Colab, set `PROJECT_ROOT` accordingly.
- If your repository is in Google Drive, mount Drive first and point `PROJECT_ROOT` to that folder.


In [1]:
# Optional: mount Drive if your repo is stored there
#  mount Drive if using Colab
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
from pathlib import Path

# Update this path to your repository root in Colab
PROJECT_ROOT = Path('/content/drive/My Drive/AutoResearch/AutoResearch-Copilot')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'PROJECT_ROOT not found: {PROJECT_ROOT}. Update the path in this cell.'
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Working directory:', PROJECT_ROOT)


Mounted at /content/drive
Working directory: /content/drive/My Drive/AutoResearch/AutoResearch-Copilot


## 1) Define paths

In [2]:
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'h1-negation-contrast-diagnostics' / 'results'
SEED_CSV = PROJECT_ROOT / 'data' / 'absa_rts_seed_restaurants.csv'
EVAL_CSV = RESULTS_DIR / 'absa_rts_eval.csv'
PAIRS_CSV = RESULTS_DIR / 'absa_rts_pairs.csv'
A1_PRED_CSV = RESULTS_DIR / 'a1_predictions.csv'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results dir:', RESULTS_DIR)
print('Seed csv:', SEED_CSV)


Results dir: /content/drive/My Drive/AutoResearch/AutoResearch-Copilot/experiments/h1-negation-contrast-diagnostics/results
Seed csv: /content/drive/My Drive/AutoResearch/AutoResearch-Copilot/data/absa_rts_seed_restaurants.csv


## 2) Build ABSA-RTS (negation + contrast)

In [3]:
!PYTHONPATH=src python -m absa_rts.build_rts \
  --input data/absa_rts_seed_restaurants.csv \
  --output-dir experiments/h1-negation-contrast-diagnostics/results \
  --domain restaurants

Generated 19 pairs and 38 evaluation rows in experiments/h1-negation-contrast-diagnostics/results


## 3) Generate A1 predictions (replace later with your real model outputs)

This uses a placeholder predictor + noise to simulate an imperfect baseline file.
Set `NOISE=0.0` if you want deterministic predictions from the simple heuristic.

In [4]:
NOISE = 0.10
SEED = 17

!PYTHONPATH=src python -m absa_rts.generate_predictions \
  --eval-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_eval.csv \
  --output-csv experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv \
  --noise {NOISE} \
  --seed {SEED}

Wrote 38 predictions to experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv


## 4) Evaluate A0 vs A1

In [5]:
!PYTHONPATH=src python -m absa_rts.evaluate_baselines \
  --eval-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_eval.csv \
  --pairs-csv experiments/h1-negation-contrast-diagnostics/results/absa_rts_pairs.csv \
  --a1-predictions experiments/h1-negation-contrast-diagnostics/results/a1_predictions.csv \
  --output-dir experiments/h1-negation-contrast-diagnostics/results

Wrote metrics to experiments/h1-negation-contrast-diagnostics/results


## 5) Inspect outputs

In [6]:
import json
from pathlib import Path

summary_path = Path('experiments/h1-negation-contrast-diagnostics/results/baseline_summary.csv')
metrics_path = Path('experiments/h1-negation-contrast-diagnostics/results/baseline_metrics.json')

print(summary_path.read_text(encoding='utf-8'))
print('\nDetailed metrics:\n')
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2))


model,accuracy,macro_f1,metamorphic_pass_rate
A0,0.7632,0.8322,0.5263
A1,0.6842,0.6551,0.5263


Detailed metrics:

{
  "models": [
    {
      "model": "A0",
      "accuracy": 0.7631578947368421,
      "macro_f1": 0.8321678321678321,
      "metamorphic_pass_rate": 0.5263157894736842,
      "category_metrics": {
        "negation": {
          "accuracy": 0.5,
          "macro_f1": 0.3323013415892673,
          "metamorphic_pass_rate": 0.0
        },
        "contrast": {
          "accuracy": 1.0,
          "macro_f1": 1.0,
          "metamorphic_pass_rate": 1.0
        }
      }
    },
    {
      "model": "A1",
      "accuracy": 0.6842105263157895,
      "macro_f1": 0.6551226551226551,
      "metamorphic_pass_rate": 0.5263157894736842,
      "category_metrics": {
        "negation": {
          "accuracy": 0.4444444444444444,
          "macro_f1": 0.30501089324618735,
          "metamorphic_pass_rate": 0.0
        },
        "contrast": {
          "accuracy": 0.9,
          "macro_f

## 6) (Optional) Export result files to Drive

Uncomment and set your destination path if needed.

In [ ]:
# import shutil
# target = Path('/content/drive/MyDrive/absa_h1_results')
# target.mkdir(parents=True, exist_ok=True)
# for p in RESULTS_DIR.glob('*'):
#     if p.is_file():
#         shutil.copy2(p, target / p.name)
# print('Copied files to', target)
